In [ ]:
#!/usr/bin/env python3
"""
День 4. WGBS/Bismark: анализ метилирования и пересечения с эпигеномными треками
Полный пайплайн для анализа данных метилирования MoPh7
"""

import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyBigWig
import pysam
import gzip
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("/home/varov/OMICS_course_spring_2026/day4_WGBS_practice")
os.chdir(BASE_DIR)

DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
BISMARK_DIR = DATA_DIR / "bismark"
REFERENCE_DIR = DATA_DIR / "reference"
TRACKS_DIR = RESULTS_DIR / "tracks"
TABLES_DIR = RESULTS_DIR / "tables"
MACS_DIR = RESULTS_DIR / "macs"

# создаем  папки
for d in [BISMARK_DIR, REFERENCE_DIR, TRACKS_DIR, TABLES_DIR, MACS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# скачиваем файл с метилированием
bismark_url = "https://genedev.bionet.nsc.ru/ftp/by_User/DashaPanchenko/OMICS_course_spring_2026/day4/bismark_output/MoPh7/MoPh7_1_bismark_bt2_pe.bismark.cov.gz"
bismark_file = BISMARK_DIR / "MoPh7_1_bismark_bt2_pe.bismark.cov.gz"

if not bismark_file.exists():
    print(f"Скачивание: {bismark_url}")
    subprocess.run([
        "wget", "--no-check-certificate",
        "-O", str(bismark_file),
        bismark_url
    ], check=True)
else:
    print(f"Файл уже существует: {bismark_file}")

report_files = {
    "MoPh7_1_bismark_bt2_PE_report.txt": "MoPh7_1_bismark_bt2_PE_report.txt",
    "MoPh7_1_bismark_bt2_pe.CpG_report.txt": "MoPh7_1_bismark_bt2_pe.CpG_report.txt",
    "MoPh7_1_bismark_bt2_pe.M-bias.txt": "MoPh7_1_bismark_bt2_pe.M-bias.txt",
    "MoPh7_1_bismark_bt2_pe.bedGraph.gz": "MoPh7_1_bismark_bt2_pe.bedGraph.gz",
    "MoPh7_1_bismark_bt2_pe.cytosine_context_summary.txt": "MoPh7_1_bismark_bt2_pe.cytosine_context_summary.txt",
    "MoPh7_1_bismark_bt2_pe_splitting_report.txt": "MoPh7_1_bismark_bt2_pe_splitting_report.txt"
}

for filename, localname in report_files.items():
    file_path = BISMARK_DIR / localname
    if not file_path.exists():
        url = f"https://genedev.bionet.nsc.ru/ftp/by_User/DashaPanchenko/OMICS_course_spring_2026/day4/bismark_output/MoPh7/{filename}"
        print(f"Скачивание: {filename}")
        subprocess.run([
            "wget", "--no-check-certificate",
            "-O", str(file_path),
            url
        ], check=True)


# проверяю наличие генома и chrom.sizes
genome_fa = REFERENCE_DIR / "T2T_human.fna"
chrom_sizes_file = REFERENCE_DIR / "chrom.sizes"

if not genome_fa.exists():
    print("Скачивание генома T2T...")
    subprocess.run([
        "wget", "-O", str(REFERENCE_DIR / "T2T_human.fna.gz"),
        "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/009/914/755/GCF_009914755.1_T2T-CHM13v2.0/GCF_009914755.1_T2T-CHM13v2.0_genomic.fna.gz"
    ], check=True)
    subprocess.run(["gzip", "-dkf", str(REFERENCE_DIR / "T2T_human.fna.gz")], check=True)
    subprocess.run(["python3", "rename_chroms_t2t.py"], check=True)

if not chrom_sizes_file.exists():
    print("Создание chrom.sizes...")
    subprocess.run(["samtools", "faidx", str(genome_fa)], check=True)
    fai_file = REFERENCE_DIR / "T2T_human.fna.fai"
    with open(fai_file) as f, open(chrom_sizes_file, 'w') as out:
        for line in f:
            parts = line.strip().split()
            out.write(f"{parts[0]}\t{parts[1]}\n")


columns = ["chrom", "start", "end", "meth_percent", "meth_count", "unmeth_count"]
nrows = 1_000_000  # Для быстрого анализа выбрано 1 млн строк

print(f"Чтение {bismark_file} (первые {nrows} строк)...")
df = pd.read_csv(
    bismark_file,
    sep="\t",
    names=columns,
    compression="gzip",
    nrows=nrows
)

print(f"Загружено {len(df):,} CpG позиций")

df["coverage"] = df["meth_count"] + df["unmeth_count"]
df["beta_value"] = df["meth_count"] / df["coverage"]
df["m_value"] = np.log2((df["meth_count"] + 1) / (df["unmeth_count"] + 1))

print("\nСтатистика:")
print(df[["coverage", "beta_value", "m_value"]].describe())

# фильтрация по покрытию

min_coverage = 5
max_coverage = df["coverage"].quantile(0.99)

filtered = df[
    (df["coverage"] >= min_coverage) &
    (df["coverage"] <= max_coverage)
].copy()

print(f"CpG до фильтрации: {len(df):,}")
print(f"CpG после фильтрации: {len(filtered):,} ({len(filtered)/len(df)*100:.1f}%)")
print(f"Максимальное покрытие (99-й перцентиль): {max_coverage:.0f}")

# сохраняем отфильтрованную таблицу
output_table = TABLES_DIR / "MoPh7_cpg_methylation_values.tsv.gz"
filtered.to_csv(output_table, sep="\t", index=False, compression="gzip")
print(f"Сохранено: {output_table}")

#  Построение QC-графиков

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Coverage
sns.histplot(filtered["coverage"], bins=50, ax=axes[0, 0])
axes[0, 0].set_xlabel("Coverage")
axes[0, 0].set_ylabel("Number of CpGs")
axes[0, 0].set_title("MoPh7: Coverage distribution")

# Beta-value
sns.histplot(filtered["beta_value"], bins=80, ax=axes[0, 1])
axes[0, 1].set_xlabel("Beta-value")
axes[0, 1].set_ylabel("Number of CpGs")
axes[0, 1].set_title("MoPh7: Beta-value distribution")

# M-value
sns.histplot(filtered["m_value"], bins=80, ax=axes[1, 0])
axes[1, 0].set_xlabel("M-value")
axes[1, 0].set_ylabel("Number of CpGs")
axes[1, 0].set_title("MoPh7: M-value distribution")

# Beta vs Coverage
sns.scatterplot(data=filtered.sample(10000), x="coverage", y="beta_value", alpha=0.3, ax=axes[1, 1])
axes[1, 1].set_xlabel("Coverage")
axes[1, 1].set_ylabel("Beta-value")
axes[1, 1].set_title("MoPh7: Beta-value vs Coverage")

plt.tight_layout()
plt.savefig(TRACKS_DIR / "MoPh7_QC_plots.png", dpi=150)
print(f"Графики сохранены: {TRACKS_DIR / 'MoPh7_QC_plots.png'}")
plt.show()

# Создание bigWig треков для IGV

# Подготовка данных для bigWig
df_bw = filtered.copy()
df_bw["bw_start"] = (df_bw["start"] - 1).clip(lower=0).astype(int)
df_bw["bw_end"] = df_bw["bw_start"] + 1

# Группировка по позиции (усреднение)
df_bw = (
    df_bw.groupby(["chrom", "bw_start", "bw_end"], as_index=False)
    .agg({"beta_value": "mean", "m_value": "mean", "coverage": "mean"})
)

def write_bigwig(table, value_col, output_path, chrom_sizes_path):
    """Создание bigWig файла из таблицы"""
    table = table[["chrom", "bw_start", "bw_end", value_col]].copy()
    table = table.replace([np.inf, -np.inf], np.nan).dropna()
    table = table.sort_values(["chrom", "bw_start", "bw_end"])
    
    # анализ chrom.sizes
    chrom_sizes = {}
    with open(chrom_sizes_path) as f:
        for line in f:
            chrom, size = line.strip().split("\t")[:2]
            chrom_sizes[chrom] = int(size)
    
    bw = pyBigWig.open(str(output_path), "w")
    used_chroms = [chrom for chrom in chrom_sizes if chrom in set(table["chrom"])]
    bw.addHeader([(chrom, chrom_sizes[chrom]) for chrom in used_chroms])
    
    for chrom, part in table.groupby("chrom", sort=False):
        bw.addEntries(
            [chrom] * len(part),
            part["bw_start"].astype(int).tolist(),
            ends=part["bw_end"].astype(int).tolist(),
            values=part[value_col].astype(float).tolist(),
        )
    bw.close()
    return output_path

# Создаем треки
tracks = {
    "beta_methylation": "beta_value",
    "m_value": "m_value",
    "coverage": "coverage",
}

for track_name, column in tracks.items():
    bw_path = TRACKS_DIR / f"MoPh7_{track_name}.bw"
    write_bigwig(df_bw, column, bw_path, chrom_sizes_file)
    print(f"Создан: {bw_path}")

# Расчет GC content и CpG observed/expected

bin_size = 100
fasta = pysam.FastaFile(str(genome_fa))

regions = (
    df_bw.groupby("chrom")
    .agg(start=("bw_start", "min"), end=("bw_end", "max"))
    .reset_index()
)

gc_rows = []
for _, row in regions.iterrows():
    chrom = row["chrom"]
    region_start = int(row["start"] // bin_size * bin_size)
    region_end = int(np.ceil(row["end"] / bin_size) * bin_size)
    
    # Читаем chrom.sizes
    chrom_sizes = {}
    with open(chrom_sizes_file) as f:
        for line in f:
            chrom_name, size = line.strip().split("\t")[:2]
            chrom_sizes[chrom_name] = int(size)
    
    region_end = min(region_end, chrom_sizes.get(chrom, region_end))
    
    for start in range(region_start, region_end, bin_size):
        end = min(start + bin_size, chrom_sizes.get(chrom, region_end))
        seq = fasta.fetch(chrom, start, end).upper()
        length = len(seq)
        if length == 0:
            continue
        
        c_count = seq.count("C")
        g_count = seq.count("G")
        cg_count = seq.count("CG")
        
        gc_content = (c_count + g_count) / length
        expected_cpg = (c_count * g_count) / length if length else 0
        cpg_oe = cg_count / expected_cpg if expected_cpg > 0 else np.nan
        
        gc_rows.append((chrom, start, end, gc_content, cpg_oe))

gc_df = pd.DataFrame(gc_rows, columns=["chrom", "bw_start", "bw_end", "gc_content", "cpg_obs_exp"])

# Сохраняем как bigWig
for track_name, column in {"gc_content": "gc_content", "cpg_obs_exp": "cpg_obs_exp"}.items():
    bw_path = TRACKS_DIR / f"T2T_{track_name}_100bp.bw"
    write_bigwig(gc_df, column, bw_path, chrom_sizes_file)
    print(f"Создан: {bw_path}")

# ====================== 8. ИНТЕГРАЦИЯ С ChIP-seq ======================

# Пути к ChIP-seq трекам
chip_tracks = {
    "H3K27Ac": MACS_DIR / "MoPh7_H3K27Ac_FE.bw",
    "H3K9me3": MACS_DIR / "MoPh7_H3K9me3_FE.bw",
}

def mean_bigwig_signal(bw, chrom, start, end, window=500):
    """Извлечение среднего сигнала из bigWig вокруг позиции"""
    center = (start + end) // 2
    region_start = max(0, center - window // 2)
    region_end = center + window // 2
    try:
        values = bw.values(chrom, region_start, region_end)
        if values and len(values) > 0:
            return np.nanmean([v for v in values if not np.isnan(v)])
        return np.nan
    except:
        return np.nan

# Загружаем таблицу метилирования
df_integrate = pd.read_csv(output_table, sep="\t")

window = 500
for track_name, track_path in chip_tracks.items():
    if not track_path.exists():
        print(f"Предупреждение: файл не найден {track_path}")
        continue
    
    print(f"Извлечение сигнала из {track_name}...")
    signal_values = []
    with pyBigWig.open(str(track_path)) as bw:
        for row in df_integrate.itertuples(index=False):
            signal_values.append(
                mean_bigwig_signal(bw, row.chrom, row.start, row.end, window=window)
            )
    df_integrate[f"{track_name}_signal"] = signal_values

# Очистка от NaN
signal_columns = [f"{name}_signal" for name in chip_tracks if f"{name}_signal" in df_integrate.columns]
df_integrate = df_integrate.dropna(subset=signal_columns).copy()

# Сохраняем результаты
output_signals = TABLES_DIR / "MoPh7_methylation_chipseq_signals.tsv.gz"
df_integrate.to_csv(output_signals, sep="\t", index=False, compression="gzip")
print(f"Сохранено: {output_signals}")

# Корреляции
if len(signal_columns) > 0:
    corr_dict = {}
    for col in signal_columns:
        corr = df_integrate["beta_value"].corr(df_integrate[col])
        corr_dict[col] = corr
        print(f"Корреляция beta_value с {col}: {corr:.3f}")
    
    # Сохраняем корреляции
    corr_df = pd.DataFrame([corr_dict])
    corr_df.to_csv(TABLES_DIR / "MoPh7_methylation_chipseq_correlations.tsv", sep="\t", index=False)
    print(f"Корреляции сохранены")
